# Day 17 — Feature engineering: encoding, scaling and derived features

Run each cell with **Shift + Enter**, from the top. We use `student_data_150.csv`: 8 real students and 142 **synthetic** ones.

## 1. Introduction: why feature engineering matters

In [ ]:
# Get the student dataset: Colab starts with an empty folder. If the download fails, upload student_data_150.csv
# with the Files panel (the folder icon on the left) and run this cell again
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-017/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
import pandas as pd

# The lesson's five students: raw features
df = pd.DataFrame({"study_hours": [2, 4, 6, 3, 8], "attendance": [55, 70, 90, 65, 95],
                   "prev_score": [40, 55, 85, 50, 92], "assignments_done": [3, 5, 9, 4, 10]})
print(df.to_string(index=False))

 study_hours  attendance  prev_score  assignments_done
           2          55          40                 3
           4          70          55                 5
           6          90          85                 9
           3          65          50                 4
           8          95          92                10


In [ ]:
# Feature creation and binning
df["weekly_study_hours"] = df["study_hours"] * 7
df["attendance_category"] = pd.cut(df["attendance"], bins=[0, 60, 80, 100], labels=["Low", "Medium", "High"], include_lowest=True)
df["score_category"] = pd.cut(df["prev_score"], bins=[0, 50, 70, 100], labels=["Low", "Medium", "High"], include_lowest=True)
df["assignment_completion"] = df["assignments_done"] / 10          # completion ratio, assuming 10 assignments were set
print(df.to_string(index=False))

 study_hours  attendance  prev_score  assignments_done  weekly_study_hours attendance_category score_category  assignment_completion
           2          55          40                 3                  14                 Low            Low                    0.3
           4          70          55                 5                  28              Medium         Medium                    0.5
           6          90          85                 9                  42                High           High                    0.9
           3          65          50                 4                  21              Medium            Low                    0.4
           8          95          92                10                  56                High           High                    1.0


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

# Model A vs Model B on our 127 training students: does engineering features help? (pass/fail, 5-fold CV)
data = pd.read_csv("student_data_150.csv")
train_val, test = train_test_split(data, test_size=0.15, random_state=42, stratify=data["Passed"])
y = train_val["Passed"].astype(int)
X = train_val[["StudyHours", "Attendance", "PreviousScore"]].copy()
X["WeeklyStudyHours"] = X["StudyHours"] * 7
X["AttendanceCategory"] = pd.cut(X["Attendance"], [0, 60, 80, 100], labels=["Low", "Medium", "High"], include_lowest=True).astype(str)
X["ScoreCategory"] = pd.cut(X["PreviousScore"], [0, 50, 70, 100], labels=["Low", "Medium", "High"], include_lowest=True).astype(str)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
def model(numeric, categorical=()):
    prep = ColumnTransformer([("num", StandardScaler(), list(numeric))] +
                             ([("cat", OneHotEncoder(handle_unknown="ignore"), list(categorical))] if categorical else []))
    return make_pipeline(prep, LogisticRegression(max_iter=1000))
raw = ["StudyHours", "Attendance", "PreviousScore"]
for name, num, cat in [("A: raw features", raw, ()),
                       ("A + weekly study hours", raw + ["WeeklyStudyHours"], ()),
                       ("Categories only (binned)", ["StudyHours"], ["AttendanceCategory", "ScoreCategory"]),
                       ("Raw + categories", raw, ["AttendanceCategory", "ScoreCategory"])]:
    s = cross_val_score(model(num, cat), X, y, cv=cv)
    print(f"{name:<26} CV accuracy {s.mean():.3f} +/- {s.std():.3f}")

A: raw features            CV accuracy 0.796 +/- 0.062
A + weekly study hours     CV accuracy 0.788 +/- 0.073
Categories only (binned)   CV accuracy 0.781 +/- 0.089
Raw + categories           CV accuracy 0.796 +/- 0.062


In [ ]:
# Data leakage: a feature that would not exist at prediction time. ExamScore decides pass/fail, so a "feature" built
# from it looks perfect in testing and is useless in practice (we never know it before the exam)
X_leak = X[raw].assign(ScoreChange=train_val["ExamScore"] - train_val["PreviousScore"])
for name, cols, frame in [("Raw features", raw, X), ("Raw + ScoreChange (leaks)", raw + ["ScoreChange"], X_leak)]:
    s = cross_val_score(model(cols), frame, y, cv=cv)
    print(f"{name:<26} CV accuracy {s.mean():.3f}")

Raw features               CV accuracy 0.796


Raw + ScoreChange (leaks)  CV accuracy 0.937


## 2. Categorical encoding: label, one-hot and ordinal

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Label encoding: each category gets a number, in alphabetical order
data = ["Male", "Female", "Female", "Male"]
encoder = LabelEncoder()
encoded = encoder.fit_transform(data)
print(encoded)
print(encoder.classes_)

# A target label: alphabetical order makes Fail 0 and Pass 1
results = ["Pass", "Fail", "Pass", "Fail", "Pass"]
y_encoder = LabelEncoder()
print(y_encoder.fit_transform(results), y_encoder.classes_)

[1 0 0 1]
['Female' 'Male']
[1 0 1 0 1] ['Fail' 'Pass']


In [ ]:
# One-hot encoding with pandas: one 0/1 column per city
df = pd.DataFrame({"student": ["A", "B", "C", "D"],
                   "city": ["Hyderabad", "Mumbai", "Delhi", "Hyderabad"]})
encoded_df = pd.get_dummies(df, columns=["city"], dtype=int)
print(encoded_df.to_string(index=False))      # columns come out in alphabetical order

student  city_Delhi  city_Hyderabad  city_Mumbai
      A           0               1            0
      B           0               0            1
      C           1               0            0
      D           0               1            0


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Ordinal encoding: we state the order, Low < Medium < High
levels = [["Low"], ["Medium"], ["High"], ["Medium"]]
encoder = OrdinalEncoder(categories=[["Low", "Medium", "High"]])
print(encoder.fit_transform(levels).ravel())

# Without the order, categories are sorted alphabetically: High 0, Low 1, Medium 2 (wrong order)
print(OrdinalEncoder().fit_transform(levels).ravel())

[0. 1. 2. 1.]
[1. 2. 0. 2.]


In [ ]:
# Our dataset: Class is a nominal category (no order), with 5 students recorded as "Unknown"
train_x = train_val[["StudyHours", "Attendance", "PreviousScore", "Class"]].replace({"Class": {"Unknown": np.nan}})
print(train_x["Class"].value_counts(dropna=False).to_string())
print("Pass rate by class:", y.groupby(train_x["Class"]).mean().round(2).to_dict())

# Fit the encoder on training data only; a class never seen in training ("D") becomes all zeros
onehot = OneHotEncoder(handle_unknown="ignore").fit(train_x[["Class"]].dropna())
print(onehot.categories_[0])
print(onehot.transform(pd.DataFrame({"Class": ["B", "D"]})).toarray())
try:
    OneHotEncoder().fit(train_x[["Class"]].dropna()).transform(pd.DataFrame({"Class": ["D"]}))
except ValueError as error:
    print("Without handle_unknown='ignore':", str(error)[:60])

Class
A      43
B      42
C      37
NaN     5
Pass rate by class: {'A': 0.72, 'B': 0.74, 'C': 0.68}
['A' 'B' 'C']
[[0. 1. 0.]
 [0. 0. 0.]]
Without handle_unknown='ignore': Found unknown categories ['D'] in column 0 during transform


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# A production-style pipeline: impute and scale numbers, impute and one-hot encode categories, then a model.
# Every step is fitted on the training folds only, so nothing leaks from the fold being scored
numeric_features = ["StudyHours", "Attendance", "PreviousScore"]
categorical_features = ["Class"]
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                 ("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("numeric", numeric_pipeline, numeric_features),
                                  ("categorical", categorical_pipeline, categorical_features)])
pipe = Pipeline([("preprocessor", preprocessor),
                 ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))])
with_class = cross_val_score(pipe, train_x, y, cv=cv)
without_class = cross_val_score(pipe.set_params(preprocessor__categorical="drop"), train_x, y, cv=cv)
print(f"With Class    CV accuracy {with_class.mean():.3f}")
print(f"Without Class CV accuracy {without_class.mean():.3f}")

# Fit once on all 127 training students and score two new students, one from a class never seen
pipe.set_params(preprocessor__categorical=categorical_pipeline).fit(train_x, y)
new = pd.DataFrame({"StudyHours": [5, 5], "Attendance": [85, 85], "PreviousScore": [70, 70], "Class": ["B", "D"]})
print("P(pass):", pipe.predict_proba(new)[:, 1].round(2))

With Class    CV accuracy 0.780
Without Class CV accuracy 0.804
P(pass): [1.   0.99]


## 3. Feature scaling: standardization, normalization and robust scaling

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# Three scalers on five students: each column is scaled on its own
df = pd.DataFrame({"study_hours": [2, 4, 6, 8, 10],
                   "attendance": [50, 60, 75, 85, 95],
                   "family_income": [20000, 30000, 45000, 60000, 90000]})
print(df.to_string(index=False))
for scaler in [StandardScaler(), MinMaxScaler(), RobustScaler()]:
    scaled_df = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
    print(f"\n{type(scaler).__name__}")
    print(scaled_df.round(3).to_string(index=False))

 study_hours  attendance  family_income
           2          50          20000
           4          60          30000
           6          75          45000
           8          85          60000
          10          95          90000

StandardScaler
 study_hours  attendance  family_income
      -1.414      -1.410         -1.180
      -0.707      -0.797         -0.773
       0.000       0.123         -0.163
       0.707       0.736          0.448
       1.414       1.349          1.668

MinMaxScaler
 study_hours  attendance  family_income
        0.00       0.000          0.000
        0.25       0.222          0.143
        0.50       0.556          0.357
        0.75       0.778          0.571
        1.00       1.000          1.000

RobustScaler
 study_hours  attendance  family_income
        -1.0        -1.0         -0.833
        -0.5        -0.6         -0.500
         0.0         0.0          0.000
         0.5         0.4          0.500
         1.0         0.8          1.

In [ ]:
# The impact of an outlier: 1000 among 10, 20, 30, 40
values = np.array([[10], [20], [30], [40], [1000]])
print("mean", values.mean(), "| std", values.std().round(1), "| median", np.median(values),
      "| IQR", np.percentile(values, 75) - np.percentile(values, 25))
for name, scaler in {"Standard": StandardScaler(), "MinMax": MinMaxScaler(), "Robust": RobustScaler()}.items():
    print(f"{name:<9}", scaler.fit_transform(values).ravel().round(2))

mean 220.0 | std 390.1 | median 30.0 | IQR 20.0
Standard  [-0.54 -0.51 -0.49 -0.46  2.  ]
MinMax    [0.   0.01 0.02 0.03 1.  ]
Robust    [-1.  -0.5  0.   0.5 48.5]


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Does scaling matter on our data? Same 127 training students, same 5 folds, with and without StandardScaler
features = ["StudyHours", "Attendance", "PreviousScore"]
print(train_val[features].agg(["min", "max"]).to_string())
for name, clf in [("KNN (k=5)", KNeighborsClassifier()),
                  ("Logistic regression", LogisticRegression(max_iter=1000)),
                  ("Decision tree", DecisionTreeClassifier(max_depth=3, random_state=42))]:
    raw_cv = cross_val_score(clf, train_val[features], y, cv=cv).mean()
    scaled_cv = cross_val_score(make_pipeline(StandardScaler(), clf), train_val[features], y, cv=cv).mean()
    print(f"{name:<20} unscaled {raw_cv:.3f} | scaled {scaled_cv:.3f}")

# Why does the tree change at all? Scaling keeps the order, so the splits are the same; look at the students it changed
from sklearn.model_selection import cross_val_predict
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
raw_pred = cross_val_predict(tree, train_val[features], y, cv=cv)
scaled_pred = cross_val_predict(make_pipeline(StandardScaler(), tree), train_val[features], y, cv=cv)
print("Tree predictions that differ:", (raw_pred != scaled_pred).sum(), "of", len(y))
print(train_val[features][raw_pred != scaled_pred].to_string(index=False))

     StudyHours  Attendance  PreviousScore
min         0.5        55.0             27
max         7.0       100.0             99


KNN (k=5)            unscaled 0.772 | scaled 0.795
Logistic regression  unscaled 0.796 | scaled 0.796


Decision tree        unscaled 0.717 | scaled 0.733


Tree predictions that differ: 2 of 127
 StudyHours  Attendance  PreviousScore
        4.5        86.0             44
        4.5        79.0             48


In [ ]:
# Leakage: fitting the scaler on all 127 students before cross-validation lets each fold's scoring students shape it
leaky_X = StandardScaler().fit_transform(train_val[features])
leaky = cross_val_score(KNeighborsClassifier(), leaky_X, y, cv=cv).mean()
model = Pipeline([("scaler", StandardScaler()), ("classifier", KNeighborsClassifier())])
correct = cross_val_score(model, train_val[features], y, cv=cv).mean()
print(f"Scaler fitted before splitting: {leaky:.3f} | inside the pipeline: {correct:.3f}")

# The pipeline learns the scaler's mean and spread from the training data it is fitted on
model.fit(train_val[features], y)
print("Learned means:", model.named_steps["scaler"].mean_.round(2))
print("Learned stds: ", model.named_steps["scaler"].scale_.round(2))

Scaler fitted before splitting: 0.771 | inside the pipeline: 0.795
Learned means: [ 3.61 79.13 62.39]
Learned stds:  [ 1.37 10.   15.11]
